In [1]:
!pip install xgboost lightgbm catboost joblib --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.7 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import joblib
import xgboost as xgb
import warnings
warnings.filterwarnings("ignore")

In [3]:
df_main = pd.read_csv("/content/weekly_features_engineered_v2.csv")
df_main["Date"] = pd.to_datetime(df_main["Date"])
df_main = df_main.sort_values("Date")
df_main.replace([np.inf, -np.inf], np.nan, inplace=True)
df_main = df_main.dropna().reset_index(drop=True)

print("Dataset shape:", df_main.shape)
print("Date range:", df_main["Date"].min(), "→", df_main["Date"].max())
print("Vegetables:", df_main["Vegetable"].unique().tolist())

Dataset shape: (4747, 63)
Date range: 2010-01-11 00:00:00 → 2025-12-22 00:00:00
Vegetables: ['Cabbage', 'Tomatoes', 'Pumpkin', 'Carrot', 'Brinjals', 'Bitter Gourd']


In [4]:
# Load best tuned model saved from tuning notebook
cat_model_tuned = joblib.load("/content/cat_tuned-new.pkl")

best_model      = cat_model_tuned
best_model_name = "CatBoost"

print("Model loaded:", best_model_name)

Model loaded: CatBoost


In [5]:
# Arosha's wholesale predictions — Week 9, one row per vegetable
arosha_pred = pd.read_csv("/content/predictions_week9_2026-wholesale.csv")
arosha_pred['week_num'] = arosha_pred['Week'].str.replace('W', '').astype(int)

# Amika's weather predictions — daily, Feb 26 – Mar 4 (Week 9), 7 cities
amika_pred = pd.read_csv("/content/upcoming_7_days-weather.csv")
amika_pred['date']     = pd.to_datetime(amika_pred['date'])
amika_pred['week_num'] = 9  # all rows are Week 9

# Aggregate all 7 cities and 7 days to a single weekly average
amika_weekly = amika_pred.groupby('week_num').agg(
    avg_flood_prob   = ('prob_flood_risk', 'mean'),
    avg_drought_prob = ('prob_drought',    'mean')
).reset_index()

print("Arosha wholesale predictions loaded:", arosha_pred.shape)
print("Amika weather predictions loaded:", amika_pred.shape)
print("Aggregated weather for week_num:", amika_weekly['week_num'].tolist())
print(amika_weekly)

Arosha wholesale predictions loaded: (6, 5)
Amika weather predictions loaded: (49, 7)
Aggregated weather for week_num: [9]
   week_num  avg_flood_prob  avg_drought_prob
0         9        0.000763          0.000131


In [6]:
# Week 9 = Feb 26 – Mar 4, 2026
# Using Feb 26 as the representative date for Week 9
future_dates = pd.to_datetime(["2026-02-26"])
vegetables   = df_main["Vegetable"].unique()

print("Predicting for vegetables:", vegetables.tolist())
print("Predicting for dates:", future_dates.tolist())
print("Week: 9 (Feb 26 – Mar 4, 2026)")

Predicting for vegetables: ['Cabbage', 'Tomatoes', 'Pumpkin', 'Carrot', 'Brinjals', 'Bitter Gourd']
Predicting for dates: [Timestamp('2026-02-26 00:00:00')]
Week: 9 (Feb 26 – Mar 4, 2026)


In [8]:
future_rows = []

for veg in vegetables:
    veg_df = df_main[df_main["Vegetable"] == veg].sort_values("Date").copy()

    for date in future_dates:
        last_row   = veg_df.iloc[-1].copy()
        last_price = last_row["Price"] if len(future_rows) == 0 else future_rows[-1]["Predicted Price"]

        new_row         = last_row.copy()
        new_row["Date"] = date
        week_num        = 9   # fixed — Week 9 per custom format
        year            = date.year

        # Pipeline Input 1 — Arosha's wholesale prediction
        arosha_row = arosha_pred[
            (arosha_pred['Year']      == year) &
            (arosha_pred['week_num']  == week_num) &
            (arosha_pred['Vegetable'] == veg)
        ]
        if not arosha_row.empty:
            wp = arosha_row['Predicted Price'].values[0]
            new_row["Wholesale_Price"]         = wp
            new_row["Wholesale_Lag1"]          = last_row["Wholesale_Price"]
            new_row["Wholesale_Lag2"]          = last_row["Wholesale_Lag1"]
            new_row["Wholesale_Rolling_Mean4"] = (
                wp + last_row["Wholesale_Price"] +
                last_row["Wholesale_Lag1"] + last_row["Wholesale_Lag2"]
            ) / 4
        else:
            print(f"WARNING: No wholesale data found for {veg} Week {week_num} {year}")

        # Pipeline Input 2 — Amika's weather prediction
        amika_row = amika_weekly[amika_weekly['week_num'] == week_num]
        if not amika_row.empty:
            new_row["avg_flood_prob"]   = amika_row['avg_flood_prob'].values[0]
            new_row["avg_drought_prob"] = amika_row['avg_drought_prob'].values[0]

        # Roll market price lag features forward
        if "Price_Lag_4" in new_row.index: new_row["Price_Lag_4"] = last_row["Price_Lag_3"]
        if "Price_Lag_3" in new_row.index: new_row["Price_Lag_3"] = last_row["Price_Lag_2"]
        if "Price_Lag_2" in new_row.index: new_row["Price_Lag_2"] = last_row["Price_Lag_1"]
        if "Price_Lag_1" in new_row.index: new_row["Price_Lag_1"] = last_price

        # Update rolling means
        if "Rolling_Mean_4" in new_row.index:
            new_row["Rolling_Mean_4"] = (
                last_row["Price_Lag_1"] + last_row["Price_Lag_2"] +
                last_row["Price_Lag_3"] + last_price
            ) / 4
        if "Rolling_Mean_8" in new_row.index:
            new_row["Rolling_Mean_8"] = (
                new_row["Rolling_Mean_4"] + last_row["Rolling_Mean_4"]
            ) / 2

        # Update time features
        new_row["Month"]        = date.month
        new_row["Week_of_Year"] = week_num
        new_row["Year"]         = year
        new_row["Quarter"]      = (date.month - 1) // 3 + 1

        # Predict
        X_new = pd.DataFrame([new_row.drop(["Date", "Vegetable", "Price"])])
        if best_model_name == "XGBoost":
            pred_price = best_model.predict(xgb.DMatrix(X_new))[0]
        else:
            pred_price = best_model.predict(X_new)[0]

        future_rows.append({
            "Vegetable":       veg,
            "Week":            "W9",
            "Year":            year,
            "Predicted Price": round(pred_price, 2),
            "Wholesale_Input": round(new_row.get("Wholesale_Price", 0), 2),
            "Flood_Risk":      round(new_row.get("avg_flood_prob", 0), 4),
            "Drought_Risk":    round(new_row.get("avg_drought_prob", 0), 4)
        })

        new_row["Price"] = pred_price
        veg_df = pd.concat([veg_df, pd.DataFrame([new_row])], ignore_index=True)

future_output = pd.DataFrame(future_rows)
future_output = future_output.sort_values("Vegetable").reset_index(drop=True)
print(future_output.to_string())

      Vegetable Week  Year  Predicted Price  Wholesale_Input  Flood_Risk  Drought_Risk
0  Bitter Gourd   W9  2026          1046.34           375.98      0.0008        0.0001
1      Brinjals   W9  2026           553.60           167.45      0.0008        0.0001
2       Cabbage   W9  2026           400.27           170.74      0.0008        0.0001
3        Carrot   W9  2026           363.10           296.61      0.0008        0.0001
4       Pumpkin   W9  2026           329.44            90.24      0.0008        0.0001
5      Tomatoes   W9  2026           700.29           207.17      0.0008        0.0001


In [9]:
future_output.to_csv("/content/future_price_predictions-market.csv", index=False)

print("Saved: /content/future_price_predictions-market.csv")
print("Total predictions:", len(future_output))
print("Period: Week 9 of 2026 (Feb 26 – Mar 4)")
print("Vegetables:", future_output["Vegetable"].nunique())
print()
print("Pipeline inputs used:")
print("  Wholesale prices → Arosha's model output")
print("  Weather risk     → Amika's model output")

Saved: /content/future_price_predictions-market.csv
Total predictions: 6
Period: Week 9 of 2026 (Feb 26 – Mar 4)
Vegetables: 6

Pipeline inputs used:
  Wholesale prices → Arosha's model output
  Weather risk     → Amika's model output
